In [1]:
# Optional Google Colab setup; uncomment this block when running in Colab.
# import sys
# import os
# from google.colab import drive
# drive.mount('/content/drive')
# project_path = '/content/drive/My Drive/CPARMS_WMF'
# os.chdir(project_path)
# sys.path.append(project_path)

# 1. Import All Package

In [2]:
import os
import time
from datetime import datetime, timezone
import pandas as pd

from util.seed_config import configure_reproducibility
from util.feedback import to_L

from model.base_model.wmf_standard import WMF_Standard
from model.base_model.wmf_cparms import WMF_CPARMS
from model.base_model.wmf_cofactor import WMF_Cofactor
from model.signal_gen.generator_cparms import Generator_CPARMS
from model.signal_gen.generator_sppmi import build_item_sppmi_matrix
from model.base_model.mostpop import MostPop

from experiments.global_temporal_split import get_temporal_split
from experiments.hyperparams_set import generate_hyperparam_samples
from experiments.all_ranker import (
    build_user_activity_groups,
    ranking_metrics_at_k,
)

start_time = time.time()

# 2. Set Up Environment

In [3]:
EXPERIMENT_SEEDS = (42, 43, 44, 45, 46)

N_ITER_RANDOM_SEARCH = 50
METRIC_KS = (10, 20, 50, 100, 200)
USER_METRIC_GROUPS = ("interaction_1", "interaction_2", "interaction_3_plus")

SELECTION_METRIC = "ndcg"
SELECTION_K = 10

MODEL_PARAM_KEY = {
    "01 MostPop": None,
    "02 Standard-WMF": "standard_wmf",
    "03 CoFactor-WMF": "cofactor_wmf",
    "04 CPARMS-WMF": "cparms_wmf",
}

RESULTS_DIR = './results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# 3. Load Data

In [4]:
SELECTED_DATASETS = {
    '01_amz_beauty': './database/csv/dataset_amazon_lux_beauty_5_core.csv',
    '02_amz_industry': './database/csv/dataset_amazon_industry_5_core.csv',
    '03_amz_pantry': './database/csv/dataset_amazon_pantry_5_core.csv',
    '04_amz_music': './database/csv/dataset_amazon_music_5_core.csv',
    '05_amz_instruments': './database/csv/dataset_amazon_instruments_5_core.csv',
}

dataset_configs = []
dataset_eda_all = {}
for dataset_name, dataset_path in SELECTED_DATASETS.items():
    dataset_df = pd.read_csv(dataset_path)
    train_mat, val_mat, train_val_mat, test_mat, eda = get_temporal_split(
        dataset_df,
    )

    dataset_configs.append({
        'name': dataset_name,
        'train': train_mat,
        'val': val_mat,
        'train_val': train_val_mat,
        'test': test_mat,
    })
    dataset_eda_all[dataset_name] = eda

if not dataset_configs:
    raise ValueError('Select at least one dataset.')

# 4. Tune Hyperparameters

In [ ]:
best_params_by_seed = {}

for experiment_seed in EXPERIMENT_SEEDS:
    configure_reproducibility(experiment_seed)
    print(f"\n#################### SEED: {experiment_seed} ####################")

    shared_samples = generate_hyperparam_samples(
        rounds=N_ITER_RANDOM_SEARCH,
        global_seed=experiment_seed,
    )

    seed_best = {}
    best_params_by_seed[int(experiment_seed)] = seed_best

    for dataset_cfg in dataset_configs:
        dataset_name = dataset_cfg["name"]
        train_mat = dataset_cfg["train"]
        val_mat = dataset_cfg["val"]
        user_count, item_count = train_mat.shape
        print(f"\n========== DATASET: {dataset_name} ==========")

        train_L = to_L(train_mat)
        val_user_groups = build_user_activity_groups(train_L)

        for model_name, param_key in MODEL_PARAM_KEY.items():
            if param_key is None:
                continue
            seed_best[(model_name, dataset_name)] = {
                "best_score": float("-inf"), "best_params": None,
            }
        for round_idx, sample in enumerate(shared_samples):
            print(f"\n--- Round {round_idx + 1}/{N_ITER_RANDOM_SEARCH} ---")

            for model_name, param_key in MODEL_PARAM_KEY.items():
                if param_key is None:
                    continue
                params = sample[param_key]
                print(f"\nTraining {model_name}: {params}")

                if param_key == "standard_wmf":
                    model = WMF_Standard(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_mat, n_sweeps=params["n_sweeps"])
                    pred_source = model

                elif param_key == "cofactor_wmf":
                    model = WMF_Cofactor(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        gamma=params["gamma"],
                        lambda_context_rate=params["lambda_context_rate"],
                        negative_samples=params["negative_samples"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_mat, n_sweeps=params["n_sweeps"])
                    pred_source = model

                elif param_key == "cparms_wmf":
                    generator = Generator_CPARMS(
                        k_user=params["k_user"],
                        K_item=params["K_item"],
                        min_support=params["min_support"],
                        min_confidence=params["min_confidence"],
                        min_lift=params["min_lift"],
                        normalize=params["normalize"],
                        random_state=params["random_state"],
                    )
                    signal_mat = generator.fit_transform(train_mat)
                    model = WMF_CPARMS(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        gamma=params["gamma"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_mat, S=signal_mat, n_sweeps=params["n_sweeps"])
                    pred_source = model

                else:
                    raise ValueError(f"Unsupported model parameter key: {param_key}")

                results_ranking = ranking_metrics_at_k(
                    pred_source=pred_source,
                    train_mat=train_mat,
                    test_mat=val_mat,
                    ks=METRIC_KS,
                    user_groups=val_user_groups,
                )
                all_ndcg = results_ranking["all"]["ndcg"]
                ug = results_ranking["user"]
                print(
                    f"Results: NDCG@10 all={all_ndcg[10]:.6f}, "
                    f"i1={ug['interaction_1']['ndcg'][10]:.6f}, "
                    f"i2={ug['interaction_2']['ndcg'][10]:.6f}, "
                    f"i3+={ug['interaction_3_plus']['ndcg'][10]:.6f}"
                )

                score = results_ranking["all"][SELECTION_METRIC][SELECTION_K]
                tracker = seed_best[(model_name, dataset_name)]
                if score > tracker["best_score"]:
                    tracker["best_score"] = float(score)
                    tracker["best_params"] = dict(params)

                del results_ranking


#################### SEED: 42 ####################

========== DATASET: 01_amz_beauty ==========

--- Round 1/50 ---

Training 02 Standard-WMF: {'latent': 10, 'lambda_rate': 1e-05, 'n_sweeps': 20, 'alpha': 10.0, 'random_state': 42}
[Sweep 1/20] WMF_LOSS: 119887.886671 L2_SUM: 74489.187500 REG: 0.744892 TOTAL: 119888.631563
[Sweep 5/20] WMF_LOSS: 98011.960593 L2_SUM: 82812.273438 REG: 0.828123 TOTAL: 98012.788716
[Sweep 10/20] WMF_LOSS: 96953.725210 L2_SUM: 83571.742188 REG: 0.835717 TOTAL: 96954.560927
[Sweep 15/20] WMF_LOSS: 96647.694973 L2_SUM: 83821.039062 REG: 0.838210 TOTAL: 96648.533183
[Sweep 20/20] WMF_LOSS: 96513.671880 L2_SUM: 83910.992188 REG: 0.839110 TOTAL: 96514.510990
Results: NDCG@10 all=0.042662, i1=0.056866, i2=0.085588, i3+=0.029325

Training 03 CoFactor-WMF: {'latent': 10, 'lambda_rate': 1e-05, 'n_sweeps': 20, 'alpha': 10.0, 'random_state': 42, 'gamma': 1.0, 'lambda_context_rate': 1e-05, 'negative_samples': 10}
[Sweep 1/20] WMF_LOSS: 119887.900645 COFACTOR_LOSS: 11

# 5. Retrain & Evaluate on Test

In [ ]:
final_test_rows = []
for experiment_seed in EXPERIMENT_SEEDS:
    configure_reproducibility(experiment_seed)
    print(f"\n#################### [TEST] SEED: {experiment_seed} ####################")
    seed_best = best_params_by_seed[int(experiment_seed)]

    for dataset_cfg in dataset_configs:
        dataset_name = dataset_cfg["name"]
        train_val_mat = dataset_cfg["train_val"]
        test_mat = dataset_cfg["test"]
        user_count, item_count = train_val_mat.shape
        print(f"\n========== [TEST] DATASET: {dataset_name} ==========")

        train_val_L = to_L(train_val_mat)
        test_user_groups = build_user_activity_groups(train_val_L)
        for model_name, param_key in MODEL_PARAM_KEY.items():
            print(f"\n----- [TEST] MODEL: {model_name} -----")
            s_mat_runtime = 0.0
            log_params = {}
            validation_selection_score = float("nan")

            if param_key is None:
                t0 = time.perf_counter()
                model = MostPop(user_count=user_count, item_count=item_count)
                model.fit(Y=train_val_mat)
                model_runtime = (time.perf_counter() - t0) / 60.0
                pred_source = model

            else:
                tracker = seed_best[(model_name, dataset_name)]
                params = tracker["best_params"]
                if params is None:
                    print(f"[Skip] No valid best parameters for {model_name}")
                    continue
                print(f"{model_name} hyperparams: {params}")
                validation_selection_score = tracker["best_score"]

                if param_key == "standard_wmf":
                    t0 = time.perf_counter()
                    model = WMF_Standard(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_val_mat, n_sweeps=params["n_sweeps"])
                    model_runtime = (time.perf_counter() - t0) / 60.0
                    pred_source = model
                    log_params = dict(params)

                elif param_key == "cofactor_wmf":
                    if params["gamma"] > 0.0:
                        t0 = time.perf_counter()
                        sppmi_mat = build_item_sppmi_matrix(
                            train_val_mat,
                            negative_samples=params["negative_samples"],
                        )
                        s_mat_runtime = (time.perf_counter() - t0) / 60.0
                    else:
                        sppmi_mat = None
                    t0 = time.perf_counter()
                    model = WMF_Cofactor(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        gamma=params["gamma"],
                        lambda_context_rate=params["lambda_context_rate"],
                        negative_samples=params["negative_samples"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_val_mat, M=sppmi_mat, n_sweeps=params["n_sweeps"])
                    model_runtime = (time.perf_counter() - t0) / 60.0
                    pred_source = model
                    log_params = dict(params)

                elif param_key == "cparms_wmf":
                    generator = Generator_CPARMS(
                        k_user=params["k_user"],
                        K_item=params["K_item"],
                        min_support=params["min_support"],
                        min_confidence=params["min_confidence"],
                        min_lift=params["min_lift"],
                        normalize=params["normalize"],
                        random_state=params["random_state"],
                    )
                    t0 = time.perf_counter()
                    signal_mat = generator.fit_transform(train_val_mat)
                    s_mat_runtime = (time.perf_counter() - t0) / 60.0
                    t0 = time.perf_counter()
                    model = WMF_CPARMS(
                        user_count=user_count,
                        item_count=item_count,
                        K=params["latent"],
                        lambda_rate=params["lambda_rate"],
                        alpha=params["alpha"],
                        gamma=params["gamma"],
                        random_state=params["random_state"],
                    )
                    model.fit(Y=train_val_mat, S=signal_mat, n_sweeps=params["n_sweeps"])
                    model_runtime = (time.perf_counter() - t0) / 60.0
                    pred_source = model
                    log_params = dict(params)

                else:
                    raise ValueError(f"Unsupported model parameter key: {param_key}")
            results_ranking = ranking_metrics_at_k(
                pred_source=pred_source,
                train_mat=train_val_mat,
                test_mat=test_mat,
                ks=METRIC_KS,
                user_groups=test_user_groups,
            )
            all_ndcg = results_ranking["all"]["ndcg"]
            ug = results_ranking["user"]
            print(
                f"Results: NDCG@10 all={all_ndcg[10]:.6f}, "
                f"i1={ug['interaction_1']['ndcg'][10]:.6f}, "
                f"i2={ug['interaction_2']['ndcg'][10]:.6f}, "
                f"i3+={ug['interaction_3_plus']['ndcg'][10]:.6f}"
            )

            row = {
                "seed": int(experiment_seed),
                "dataset_name": dataset_name,
                "model": model_name,
                "s_mat_runtime": float(s_mat_runtime),
                "wmf_runtime": float(model_runtime),
                "total_runtime": float(s_mat_runtime) + float(model_runtime),
                "n_iter_random_search": N_ITER_RANDOM_SEARCH,
                "selection_metric": SELECTION_METRIC,
                "selection_k": SELECTION_K,
                "validation_selection_score": validation_selection_score,
                "n_users_eval": int(results_ranking["n_users_eval"]),
                "n_positive_targets": int(results_ranking["n_positive_targets"]),
            }
            row.update(log_params)
            for k, v in results_ranking["all"]["ndcg"].items():
                row[f"ndcg_all@{int(k)}"] = float(v)
            for g in USER_METRIC_GROUPS:
                for k, v in results_ranking["user"][g]["ndcg"].items():
                    row[f"ndcg_user_{g}@{int(k)}"] = float(v)
            final_test_rows.append(row)

            del results_ranking

# 6. Review Results

In [ ]:
df_best_results = pd.DataFrame(final_test_rows)
df_best_results.sort_values(
    by=["seed", "dataset_name", "model"], inplace=True,
)

df_best_results[[
    'dataset_name', 'seed', 'model',
    'ndcg_all@10',
    'ndcg_user_interaction_1@10',
    'ndcg_user_interaction_2@10',
    'ndcg_user_interaction_3_plus@10',
]]

In [ ]:
aggregate_cols = ["ndcg_all@10"]
aggregate_cols.extend(f"ndcg_user_{g}@10" for g in USER_METRIC_GROUPS)
aggregate_cols.extend(["s_mat_runtime", "wmf_runtime", "total_runtime"])

grouped = df_best_results.groupby(["dataset_name", "model"])
df_seed_summary = grouped[aggregate_cols].agg(["mean", "std"])
df_seed_summary.columns = [
    f"{metric}_{'sd' if stat == 'std' else stat}"
    for metric, stat in df_seed_summary.columns
]
df_seed_summary.insert(0, "seed_count", grouped["seed"].nunique())
df_seed_summary = df_seed_summary.reset_index()

df_seed_summary

# 7. Save Outputs

In [ ]:
timestamp_str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
file_name = f'{RESULTS_DIR}/final_results_{timestamp_str}.xlsx'

ordered_ndcg_cols = []
for k in METRIC_KS:
    ordered_ndcg_cols.append(f"ndcg_all@{int(k)}")
    ordered_ndcg_cols.extend(f"ndcg_user_{g}@{int(k)}" for g in USER_METRIC_GROUPS)

other_cols = [c for c in df_best_results.columns if c not in ordered_ndcg_cols]
df_results = df_best_results[other_cols + ordered_ndcg_cols]

metric_cols = ordered_ndcg_cols + ["s_mat_runtime", "wmf_runtime", "total_runtime"]
grouped = df_best_results.groupby(["dataset_name", "model"])
df_mean_sd = grouped[metric_cols].agg(["mean", "std"])
df_mean_sd.columns = [
    f"{metric}_{'sd' if stat == 'std' else stat}"
    for metric, stat in df_mean_sd.columns
]
df_mean_sd.insert(0, "seed_count", grouped["seed"].nunique())
df_mean_sd = df_mean_sd.reset_index()

df_dataset_eda = pd.DataFrame(
    [
        {"dataset_name": dataset_name, "split": split_name, **metrics}
        for dataset_name, split_eda in dataset_eda_all.items()
        for split_name, metrics in split_eda.items()
    ]
)

with pd.ExcelWriter(file_name) as writer:
    df_results.to_excel(writer, sheet_name='results', index=False)
    df_mean_sd.to_excel(writer, sheet_name='mean_sd', index=False)
    df_dataset_eda.to_excel(writer, sheet_name='dataset_eda', index=False)
print(f"Saved: {file_name}")

end_time = time.time()
print(f"Elapsed: {end_time - start_time:.1f}s")